# 3-Digit Number Recognition with MLP
Enable GPU: Runtime > Change runtime type > T4 GPU

In [ ]:
!pip install -q ipywidgets matplotlib torch torchvision
from google.colab import output
output.enable_custom_widget_manager()
print('Ready')

In [ ]:
import torch, random
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
from collections import defaultdict
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
mnist_train = torchvision.datasets.MNIST('./data', train=True,  download=True, transform=transform)
mnist_test  = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)

def index_by_label(ds):
    idx = {d: [] for d in range(10)}
    for i, (_, label) in enumerate(ds): idx[label].append(i)
    return idx

train_idx = index_by_label(mnist_train)
test_idx  = index_by_label(mnist_test)

class ThreeDigitDataset(Dataset):
    def __init__(self, base_ds, didx, n=60000, seed=42):
        self.base=base_ds; self.didx=didx; self.n=n
        rng=np.random.default_rng(seed)
        self.digits=rng.integers(0,10,size=(n,3))
    def __len__(self): return self.n
    def __getitem__(self, i):
        d1,d2,d3=self.digits[i]
        imgs=[]
        for d in (d1,d2,d3):
            j=random.choice(self.didx[int(d)])
            img,_=self.base[j]
            imgs.append(img)
        flat=torch.cat(imgs,dim=2).view(-1)
        label=int(d1)*100+int(d2)*10+int(d3)
        return flat,label

train_ds=ThreeDigitDataset(mnist_train,train_idx,n=60000)
test_ds =ThreeDigitDataset(mnist_test, test_idx, n=10000)
train_loader=DataLoader(train_ds,batch_size=256,shuffle=True, num_workers=2,pin_memory=True)
test_loader =DataLoader(test_ds, batch_size=512,shuffle=False,num_workers=2,pin_memory=True)
print(f'Train: {len(train_ds):,}  Test: {len(test_ds):,}')

In [ ]:
fig,axes=plt.subplots(2,5,figsize=(14,5))
fig.suptitle('Sample 3-digit composites',fontsize=14,fontweight='bold')
for ax in axes.flat:
    flat,label=train_ds[random.randint(0,len(train_ds)-1)]
    ax.imshow(flat.reshape(28,84).numpy(),cmap='gray')
    ax.set_title(f'{label:03d}',fontsize=12)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
class MLP(nn.Module):
    def __init__(self,in_dim=2352,n_classes=1000):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(in_dim,1024),nn.BatchNorm1d(1024),nn.ReLU(),nn.Dropout(0.3),
            nn.Linear(1024,512),  nn.BatchNorm1d(512), nn.ReLU(),nn.Dropout(0.3),
            nn.Linear(512,256),   nn.BatchNorm1d(256), nn.ReLU(),nn.Dropout(0.2),
            nn.Linear(256,n_classes)
        )
    def forward(self,x): return self.net(x)

model=MLP().to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
EPOCHS=15
criterion=nn.CrossEntropyLoss()
optimizer=optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
scheduler=optim.lr_scheduler.OneCycleLR(optimizer,max_lr=1e-3,steps_per_epoch=len(train_loader),epochs=EPOCHS)
history={'tl':[],'ta':[],'vl':[],'va':[]}

def evaluate(loader):
    model.eval(); loss,correct,total=0,0,0
    with torch.no_grad():
        for x,y in loader:
            x,y=x.to(DEVICE),y.to(DEVICE); out=model(x)
            loss+=criterion(out,y).item()*len(y)
            correct+=(out.argmax(1)==y).sum().item(); total+=len(y)
    return loss/total,correct/total

print(f'{"Ep":>4} {"TrainLoss":>10} {"TrainAcc":>9} {"ValLoss":>9} {"ValAcc":>8}')
print('-'*47)
for epoch in range(1,EPOCHS+1):
    model.train(); rl,correct,total=0,0,0
    for x,y in train_loader:
        x,y=x.to(DEVICE),y.to(DEVICE)
        optimizer.zero_grad(); out=model(x); loss=criterion(out,y)
        loss.backward(); optimizer.step(); scheduler.step()
        rl+=loss.item()*len(y); correct+=(out.argmax(1)==y).sum().item(); total+=len(y)
    tl,ta=rl/total,correct/total; vl,va=evaluate(test_loader)
    history['tl'].append(tl); history['ta'].append(ta)
    history['vl'].append(vl); history['va'].append(va)
    print(f'{epoch:>4} {tl:>10.4f} {ta*100:>8.2f}% {vl:>9.4f} {va*100:>7.2f}%')
torch.save(model.state_dict(),'mlp_3digit.pth')
print('Done - saved mlp_3digit.pth')

In [ ]:
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(13,4))
ep=range(1,len(history['tl'])+1)
ax1.plot(ep,history['tl'],'o-',label='Train',color='#4A90D9')
ax1.plot(ep,history['vl'],'s-',label='Val',  color='#E05C5C')
ax1.set_title('Loss',fontweight='bold'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(ep,[a*100 for a in history['ta']],'o-',label='Train',color='#4A90D9')
ax2.plot(ep,[a*100 for a in history['va']],'s-',label='Val',  color='#E05C5C')
ax2.set_title('Accuracy (%)',fontweight='bold'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(alpha=0.3)
plt.suptitle('Training History',fontsize=13,fontweight='bold')
plt.tight_layout(); plt.show()
print(f'Final Val Accuracy: {history["va"][-1]*100:.2f}%')

In [ ]:
model.eval()
all_preds,all_labels=[],[]
with torch.no_grad():
    for x,y in test_loader:
        out=model(x.to(DEVICE))
        all_preds.extend(out.argmax(1).cpu().tolist())
        all_labels.extend(y.tolist())
wrong_idx=[i for i,(p,l) in enumerate(zip(all_preds,all_labels)) if p!=l]
print(f'Errors: {len(wrong_idx):,}/{len(all_labels):,} ({len(wrong_idx)/len(all_labels)*100:.1f}%)')
fig,axes=plt.subplots(2,4,figsize=(14,6))
fig.suptitle('Misclassified Samples',fontsize=13,fontweight='bold')
for ax,idx in zip(axes.flat,wrong_idx[:8]):
    flat,label=test_ds[idx]
    ax.imshow(flat.reshape(28,84).numpy(),cmap='gray')
    ax.set_title(f'True:{label:03d} Pred:{all_preds[idx]:03d}',color='#E05C5C',fontsize=10)
    ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
model.eval()

def get_digit_tensor(digit):
    j=random.choice(test_idx[digit]); img,_=mnist_test[j]; return img

def build_composite(d1,d2,d3):
    imgs=[get_digit_tensor(d) for d in (d1,d2,d3)]
    comp=torch.cat(imgs,dim=2)
    return comp.squeeze(0),comp.view(-1)

@torch.no_grad()
def predict(flat):
    out=model(flat.unsqueeze(0).to(DEVICE))
    probs=torch.softmax(out,dim=1).squeeze()
    top5v,top5i=torch.topk(probs,5)
    return out.argmax(1).item(),top5i.cpu().tolist(),top5v.cpu().tolist()

slayout=widgets.Layout(width='340px'); sstyle={'description_width':'80px'}
s1=widgets.IntSlider(value=3,min=0,max=9,description='Digit 1:',style=sstyle,layout=slayout)
s2=widgets.IntSlider(value=1,min=0,max=9,description='Digit 2:',style=sstyle,layout=slayout)
s3=widgets.IntSlider(value=4,min=0,max=9,description='Digit 3:',style=sstyle,layout=slayout)
btn_new =widgets.Button(description='New random images',   button_style='primary',layout=widgets.Layout(width='190px'))
btn_rand=widgets.Button(description='Randomise all digits',layout=widgets.Layout(width='190px'))
out_w=widgets.Output()

def render(d1,d2,d3):
    img28x84,flat=build_composite(d1,d2,d3)
    pred,top5i,top5v=predict(flat)
    correct=d1*100+d2*10+d3; ok=pred==correct
    with out_w:
        clear_output(wait=True)
        fig,(ax1,ax2)=plt.subplots(1,2,figsize=(12,3.5),gridspec_kw={'width_ratios':[1,1.6]})
        ax1.imshow(img28x84.numpy(),cmap='gray',aspect='auto')
        colour='#27AE60' if ok else '#E05C5C'
        symbol=chr(10003) if ok else chr(10007)
        ax1.set_title(f'Input:{d1}{d2}{d3}  Pred:{pred:03d} {symbol}',fontsize=13,fontweight='bold',color=colour)
        ax1.axis('off')
        lbls=[f'{v:03d}' for v in top5i]
        colors=['#27AE60' if v==correct else '#4A90D9' for v in top5i]
        bars=ax2.barh(lbls,[p*100 for p in top5v],color=colors,edgecolor='white',height=0.55)
        ax2.set_xlabel('Confidence (%)'); ax2.set_title('Top-5 predictions',fontweight='bold')
        ax2.set_xlim(0,105); ax2.invert_yaxis()
        for bar,prob in zip(bars,top5v):
            ax2.text(bar.get_width()+1,bar.get_y()+bar.get_height()/2,f'{prob*100:.1f}%',va='center',fontsize=9)
        ax2.legend(handles=[
            mpatches.Patch(color='#27AE60',label='Ground truth'),
            mpatches.Patch(color='#4A90D9',label='Other')
        ],loc='lower right',fontsize=8)
        plt.tight_layout(); plt.show()

s1.observe(lambda c:render(s1.value,s2.value,s3.value),names='value')
s2.observe(lambda c:render(s1.value,s2.value,s3.value),names='value')
s3.observe(lambda c:render(s1.value,s2.value,s3.value),names='value')
btn_new.on_click(lambda b:render(s1.value,s2.value,s3.value))
def rand_all(b): s1.value=random.randint(0,9); s2.value=random.randint(0,9); s3.value=random.randint(0,9)
btn_rand.on_click(rand_all)
header=widgets.HTML('<h3 style="font-family:monospace">3-Digit MLP - Interactive Demo</h3><p style="color:#666;font-size:13px">Use sliders to pick a 3-digit number. Model predicts from MNIST samples.</p>')
display(widgets.VBox([header,s1,s2,s3,widgets.HBox([btn_new,btn_rand]),out_w]))
render(s1.value,s2.value,s3.value)

In [ ]:
group_c=defaultdict(int); group_t=defaultdict(int)
for p,l in zip(all_preds,all_labels):
    g=(l//100)*100; group_c[g]+=int(p==l); group_t[g]+=1
groups=sorted(group_t.keys())
accs=[group_c[g]/group_t[g]*100 for g in groups]
xlabels=[f'{g:03d}-{g+99:03d}' for g in groups]
fig,ax=plt.subplots(figsize=(11,4))
bars=ax.bar(xlabels,accs,color='#4A90D9',edgecolor='white')
ax.axhline(np.mean(accs),color='#E05C5C',linestyle='--',label=f'Mean {np.mean(accs):.1f}%')
ax.set_ylim(0,110); ax.set_ylabel('Accuracy (%)'); ax.legend()
ax.set_title('Test Accuracy by Hundreds Group',fontweight='bold')
for bar,acc in zip(bars,accs):
    ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+1,f'{acc:.0f}%',ha='center',va='bottom',fontsize=8)
plt.xticks(rotation=45,ha='right'); plt.tight_layout(); plt.show()